In [25]:
%pip install transformers torch ipywidgets pandas

/bin/bash: /Users/mnelimar/code/marxists-llm-tutorial/pyven/bin/python: No such file or directory


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import transformers

from transformers import pipeline
import torch

In [9]:
transformers.enable_full_determinism( 0 )

In [10]:
model_name = "openai-community/gpt2"

In [11]:
finetuned = pipeline('text-generation', model = f"./models/{model_name.replace('/', '_')}-finetuned-causal-model/")
baseline = pipeline('text-generation', model = model_name )

In [23]:
prompt = "The purpose of companies is to"
print( finetuned(prompt)[0]['generated_text'] )
print( baseline(prompt)[0]['generated_text'] )

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The purpose of companies is to make available to the workers of different colours different methods of acquiring products; to make it possible to exchange articles of merchandise into oneanother in a way that makes the purchase or sale of one product impossible; to makethe labor
The purpose of companies is to take the best practices applied in the fields of business law and antitrust with these approaches. For example, I have an example where we were having to replace one of our own patents, which was the subject of a lawsuit back


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd

text_input = widgets.Textarea(
    placeholder='Enter prompt here...',
    layout=widgets.Layout(width='100%', height='80px')
)
run_button = widgets.Button(description='Run', button_style='primary')
output_area = widgets.Output()

def on_run(b):
    prompt = text_input.value.strip() + ' '
    if not prompt:
        return
    with output_area:
        clear_output()
        print("Running... please wait")

        rows = []
    for i in range(5):
        ft_result = finetuned(prompt)
        bl_result = baseline(prompt)
        rows.append({
            'Run': i + 1,
            'Finetuned': ft_result[0]['generated_text'],
            'Baseline': bl_result[0]['generated_text'],
        })

    df = pd.DataFrame(rows).set_index('Run')

    with output_area:
        clear_output()
        with pd.option_context('display.max_colwidth', None):
            display(df)

run_button.on_click(on_run)
display(widgets.VBox([text_input, run_button, output_area]))
